In [1]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math

import pico

import cmasher as cmr
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize

pa.settings.strict_units = False

from paicos.util import get_index_of_radial_range as RadialRange


In [2]:
simfolder, analysisfolder, info, track = pico.get_sim(3, galform='tng', zoom_factor=12, descr='')

snap = pa.Snapshot(simfolder+'output/', info.snapnums[-1], basename='snapshot')
center = snap.Cat.Group['GroupPos'][0]
radius = snap.Cat.Group['Group_R_Crit200'][0]
widths = np.ones(3)*2*radius.value

In [6]:
all_pos = np.zeros((0,3))
all_mass = np.zeros(0)
all_hsml = np.zeros(0)
all_types = np.zeros(0, dtype='int')
for parttype in range(0,5):
    idx_sphere = RadialRange(snap[f"{parttype}_Coordinates"],
                       center, 0*radius, radius)
    all_pos = np.concatenate((all_pos, snap[f"{parttype}_Coordinates"][idx_sphere]), axis=0)
    all_mass = np.concatenate((all_mass, snap[f"{parttype}_Masses"][idx_sphere]), axis=0)
    if (parttype==0):
        hsml = np.cbrt((snap[f"{parttype}_Volume"]) / (4.0 * np.pi / 3.0)).to('arepo_length')
    else:
        soft_type = snap.Parameters[f'SofteningTypeOfPartType{parttype}']
        softening_comoving = (snap.Parameters[f'SofteningComovingType{soft_type}']*snap.h)*snap.length
        softening_max = (snap.Parameters[f'SofteningMaxPhysType{soft_type}']/snap.a*snap.h)*snap.length
        hsml = np.ones(snap[f"{parttype}_Coordinates"].shape[0])*min([softening_comoving, softening_max])
    all_hsml = np.concatenate((all_hsml, hsml[idx_sphere]), axis=0)
    all_types = np.concatenate((all_types, np.ones(np.sum(idx_sphere), dtype='int')*parttype))

In [7]:

pe = tc.PotentialEnergy(snap, center, widths, 
                        all_pos, all_mass, all_hsml, npix=128)

In [ ]:
U = pe.compute_potential(angle=80)